# 21 · Pairs labelled with what the client can read

Notebooks 18 to 20 trained on notebook 17's pairs: x264 only, P frames only, and each frame labelled with x264's mean QP
over its macroblocks. The client cannot read that number. What it can read, from every encoder, is the QP in the
slice header (desktop-streamer, `stream-h264.js`), and on the host's own lines it sits about a step from the mean:

| line | mean minus slice QP, settled P frames | I frames |
| --- | --- | --- |
| x264 | +0.93, never below | +3.1 |
| NVENC | -0.80 (-1.27 on screens) | +3.9 |

And the host has a second line the pairs never saw: a Windows host with an NVIDIA card runs **NVENC**, which codes with
the loop filter on, where its x264 line (`ultrafast`) codes without. The same QP leaves different artifacts behind each.

So the pairs are made again, from notebook 17's targets, with what the client reads:

- **Both host lines**, x264 and NVENC (`degrade.ENCODERS`), as `stream-ffmpeg.js` runs them, the 100 kbit floor on
  the rate buffer included (`degrade._encoder`; notebook 17's pairs had none).
- **Labelled with the slice header**: the slice QP, whether the loop filter runs, I or P. x264's mean QP is kept beside
  it, to measure the gap, never to train on.
- **I frames too**: one a clip and encoder, at whatever QP it came. A client sees one a second, and after every join.

Each QP bin (of the slice QP) keeps one frame a clip and encoder, 8 patches a frame, so the set is about the size of
notebook 17's. Notebook 22 trains on it.

**Limits.** Targets are videos, as in notebook 17. The NVENC line here reads frames that swscale shrank with bicubic;
the host's NVENC line shrinks on the GPU (`scale_d3d11`), with a filter of its own.

Needs: notebook 17, part 5's target list (`qp/qp_targets.csv`), and an NVIDIA GPU with NVENC. Writes `qp_slice_pairs/`.

In [ ]:
"""Settings."""

import json
import os
import zlib
from concurrent.futures import ThreadPoolExecutor
from threading import BoundedSemaphore

import numpy as np
import pandas as pd
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import degrade, metrics, models, patches, sources
from helpers.training import DEVICE

TARGETS_FILE = sources.ROOT / "upscale" / "qp" / "qp_targets.csv"   # notebook 17, part 5
PAIRS_DIR = sources.ROOT / "upscale" / "qp_slice_pairs"

# notebook 17's clips and targets
SIZES = ((1920, 1080), (1280, 720))
CLIP_FRAMES, SETTLE = 32, 8
TEST_CLIPS_PER_VIDEO, TRAIN_CLIPS_PER_VIDEO = 2, 1
TARGET_MAX_QP, TARGET_MAX_ROUND_TRIP_DB = 18, 44

# the pairs
SCALE = 2
PAIR_MBPS = (0.25, 0.5, 1, 2, 4, 8)          # streams to find frames of each QP bin in
QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46, 52)   # bins of the slice QP
FRAMES_PER_BIN, INTRA_FRAMES = 1, 1          # a clip and encoder: P frames in each bin, I frames in all
PATCHES_PER_FRAME, PATCH_HR, HALO = 8, 128, 4
VAR_MIN, MIN_LR_PSNR = 24.0, 22.0            # skip flat spots, and spots with nothing of the target left
WORKERS, NVENC_AT_ONCE = 6, 3                # a consumer card runs a few NVENC sessions at once

LUMA = np.array([0.2126, 0.7152, 0.0722], np.float32)
print("datasets", sources.ROOT, " device", DEVICE)

## 1. Targets

Notebook 17's clips of YouTube video, rebuilt the way it picked them (one clip a training video, two a test video,
spread over it), and of those its targets: cheap to code and detailed at the finest scale. Test videos are validation.

In [ ]:
def video_clips(split, per_video):
    """`per_video` sequences of each video of a split, spread over it (notebook 17's `video_clips`)."""
    videos = {}
    for path in sources.files("yt8m-videos", split):
        videos.setdefault(path.parts[-3:-1], []).append(path)
    clips = []
    for (category, video), sequences in videos.items():
        step = max(1, len(sequences) // per_video)
        for path in sorted(sequences)[step // 2::step][:per_video]:
            clips.append({"key": f"yt8m-videos/{video}/{path.stem}", "category": category, "path": path,
                          "scored": split == "test"})
    return clips


measured = pd.read_csv(TARGETS_FILE)
chosen = set(measured[(measured["reference_qp"] < TARGET_MAX_QP)
                      & (measured["round_trip_db"] <= TARGET_MAX_ROUND_TRIP_DB)]["key"])
targets = [clip for clip in video_clips("test", TEST_CLIPS_PER_VIDEO) + video_clips("train", TRAIN_CLIPS_PER_VIDEO)
           if clip["key"] in chosen]
print(f"{len(targets)} targets: {sum(not c['scored'] for c in targets)} train, {sum(c['scored'] for c in targets)} validation")

## 2. Pairs

For each target, encoder and bitrate in `PAIR_MBPS`: the host's stream at x2 and the slice header report on each frame.
Then, per encoder, `FRAMES_PER_BIN` settled P frames of each slice QP bin and `INTRA_FRAMES` I frames, from whichever
stream gave them, and `PATCHES_PER_FRAME` patches of each (not flat, not destroyed). One encoder's streams are held at a
time, and only a few NVENC sessions run at once.

In [ ]:
PATCH_LR = PATCH_HR // SCALE
INPUT_LR = PATCH_LR + 2 * HALO
QP_BINS = list(zip(QP_EDGES[:-1], QP_EDGES[1:]))
nvenc_sessions = BoundedSemaphore(NVENC_AT_ONCE)


def frames_of(clip):
    """A clip's first `CLIP_FRAMES` frames, centre-cropped to the largest of `SIZES` it fills, and its frame rate."""
    frames, fps = sources.load_sequence(clip["path"])
    frames = frames[:CLIP_FRAMES]
    size = next((s for s in SIZES if frames.shape[2] >= s[0] and frames.shape[1] >= s[1]), None)
    if size is None or len(frames) < CLIP_FRAMES:
        return None, fps
    top, left = (frames.shape[1] - size[1]) // 4 * 2, (frames.shape[2] - size[0]) // 4 * 2
    return frames[:, top:top + size[1], left:left + size[0]], fps


def lr_psnr(lr_core, hr_patch):
    """Luma PSNR of a stream spot against a clean downscale of its target."""
    clean = (hr_patch.astype(np.float32) @ LUMA).reshape(PATCH_LR, SCALE, PATCH_LR, SCALE).mean(axis=(1, 3))
    mse = ((lr_core.astype(np.float32) @ LUMA - clean) ** 2).mean()
    return 10 * np.log10(255 ** 2 / max(mse, 1e-10))


def spots(hr, lr, rng):
    """Up to `PATCHES_PER_FRAME` (LR with its halo, HR) of one frame, not flat and not destroyed."""
    out = []
    for _ in range(4 * PATCHES_PER_FRAME):
        if len(out) == PATCHES_PER_FRAME:
            break
        y, x = (2 * int(rng.integers(0, (size - INPUT_LR) // 2 + 1)) for size in lr.shape[:2])   # even pixels: 4:2:0
        y_hr, x_hr = (y + HALO) * SCALE, (x + HALO) * SCALE
        hr_patch, lr_patch = hr[y_hr:y_hr + PATCH_HR, x_hr:x_hr + PATCH_HR], lr[y:y + INPUT_LR, x:x + INPUT_LR]
        if (hr_patch.astype(np.float32) @ LUMA).var() >= VAR_MIN and lr_psnr(lr_patch[HALO:-HALO, HALO:-HALO], hr_patch) >= MIN_LR_PSNR:
            out.append((lr_patch.copy(), hr_patch.copy()))
    return out


def streams_of(frames, fps, encoder):
    """{mbps: (Degradation, HR, LR, report)} of one encoder at every bitrate of `PAIR_MBPS`."""
    lr_width, lr_height = degrade.stream_size(*degrade.crop(frames, SCALE).shape[2:0:-1], SCALE)
    out = {}
    for mbps in PAIR_MBPS:
        made = degrade.Degradation(scale=SCALE, bpp=mbps * 1e6 / (lr_width * lr_height * fps), fps=fps,
                                   filter="bicubic", deblock=False, encoder=encoder)
        if encoder == "nvenc":
            with nvenc_sessions:
                out[mbps] = (made, *degrade.degrade(frames, made, with_frames=True))
        else:
            out[mbps] = (made, *degrade.degrade(frames, made, with_frames=True))
    return out


FIELDS = ("lr", "hr", "qp", "mean_qp", "deblocked", "intra", "encoder")


def pairs_of(clip):
    """One target's patches, a list per `FIELDS`, and a manifest source per stream that gave any."""
    rng = np.random.default_rng(zlib.crc32(clip["key"].encode()))
    frames, fps = frames_of(clip)
    out = {name: [] for name in FIELDS} | {"sources": []}
    if frames is None:
        return out
    for e, encoder in enumerate(degrade.ENCODERS):
        streams = streams_of(frames, fps, encoder)
        found = {qp_bin: [] for qp_bin in QP_BINS} | {"intra": []}
        for mbps, (_, _, _, _, report) in streams.items():
            for row in report:
                if row["type"] == "I":
                    found["intra"].append((mbps, row))
                elif row["frame"] >= SETTLE:
                    for low, high in QP_BINS:
                        if low <= row["slice_qp"] < high:
                            found[low, high].append((mbps, row))
        chosen = {}
        for key, candidates in found.items():
            keep = INTRA_FRAMES if key == "intra" else FRAMES_PER_BIN
            for i in rng.permutation(len(candidates))[:keep]:
                mbps, row = candidates[i]
                chosen.setdefault(mbps, []).append(row)
        for mbps, rows in sorted(chosen.items()):
            made, hr, lr, _, _ = streams[mbps]
            first = len(out["lr"])
            for row in rows:
                for lr_patch, hr_patch in spots(hr[row["frame"]], lr[row["frame"]], rng):
                    out["lr"].append(lr_patch)
                    out["hr"].append(hr_patch)
                    out["qp"].append(row["slice_qp"])
                    out["mean_qp"].append(row.get("qp", np.nan))
                    out["deblocked"].append(row["deblocked"])
                    out["intra"].append(row["type"] == "I")
                    out["encoder"].append(e)
            if len(out["lr"]) > first:
                out["sources"].append({"key": clip["key"], "category": clip["category"], "mbps": mbps, "encoder": encoder,
                                       "patches": [first, len(out["lr"])], "degradation": made.as_dict()})
        del streams
    return out


buckets = {split: {name: [] for name in FIELDS} | {"sources": []} for split in ("train", "val")}
with ThreadPoolExecutor(WORKERS) as pool:
    for done, (clip, made) in enumerate(zip(targets, pool.map(pairs_of, targets)), 1):
        bucket = buckets["val" if clip["scored"] else "train"]
        offset = len(bucket["lr"])
        bucket["sources"] += [{**s, "patches": [s["patches"][0] + offset, s["patches"][1] + offset]} for s in made["sources"]]
        for name in FIELDS:
            bucket[name] += made[name]
        if done % 50 == 0:
            print(f"{done} of {len(targets)} targets")

PAIRS_DIR.mkdir(parents=True, exist_ok=True)
DTYPES = {"qp": np.float32, "mean_qp": np.float32, "deblocked": np.bool_, "intra": np.bool_, "encoder": np.int8}
splits = {}
for split, bucket in buckets.items():
    np.save(PAIRS_DIR / f"{split}_lr.npy", np.stack(bucket["lr"])[:, None])   # (patches, 1 frame, h, w, 3)
    np.save(PAIRS_DIR / f"{split}_hr.npy", np.stack(bucket["hr"])[:, None])
    for name, dtype in DTYPES.items():
        np.save(PAIRS_DIR / f"{split}_{name}.npy", np.array(bucket[name], dtype))
    splits[split] = {"patches": len(bucket["lr"]), "lr": f"{split}_lr.npy", "hr": f"{split}_hr.npy",
                     **{name: f"{split}_{name}.npy" for name in DTYPES}, "sources": bucket["sources"]}
    print(f"{split:<6}{len(bucket['lr']):7} patches from {len({s['key'] for s in bucket['sources']})} clips")
manifest = {"scale": SCALE, "patch_lr": PATCH_LR, "patch_hr": PATCH_HR, "halo": HALO, "input_lr": INPUT_LR,
            "frames_per_patch": 1, "encoders": list(degrade.ENCODERS), "splits": splits}
(PAIRS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")

## 3. Checks

What the set holds, by encoder, frame type and slice QP bin; and bicubic against the target, which must get worse as
the slice QP rises, or the label means nothing. Then, on x264's patches, how far x264's mean is from the slice QP.

In [ ]:
manifest = json.loads((PAIRS_DIR / "manifest.json").read_text(encoding="utf-8"))
val_lr, val_hr = patches.load(PAIRS_DIR, "val", manifest)
val = {name: np.load(PAIRS_DIR / manifest["splits"]["val"][name]) for name in DTYPES}


def to_tensor(images):
    return torch.from_numpy(np.array(images)).to(DEVICE).permute(0, 3, 1, 2).float() / 255


@torch.no_grad()
def bicubic_ssim(members):
    """Mean SSIM-Y of bicubic on these validation patches."""
    out = [metrics.per_image(patches.trim(models.bicubic(to_tensor(val_lr[batch]), SCALE), HALO * SCALE),
                             to_tensor(val_hr[batch]))["ssim"].cpu() for batch in np.array_split(members, max(1, len(members) // 256))]
    return float(torch.cat(out).mean())


rows = []
for e, encoder in enumerate(manifest["encoders"]):
    for kind, is_intra in (("P", False), ("I", True)):
        for low, high in QP_BINS:
            members = np.flatnonzero((val["encoder"] == e) & (val["intra"] == is_intra) & (val["qp"] >= low) & (val["qp"] < high))
            if len(members):
                rows.append({"encoder": encoder, "type": kind, "slice QP": f"{low}-{high}", "patches": len(members),
                             "bicubic SSIM-Y": round(bicubic_ssim(members), 4),
                             "loop filter": f"{val['deblocked'][members].mean():.0%}"})
print(pd.DataFrame(rows).to_string(index=False))

train_counts = {name: np.load(PAIRS_DIR / manifest["splits"]["train"][name]) for name in ("encoder", "intra")}
print(f"\ntrain: {len(train_counts['encoder'])} patches, " + ", ".join(
    f"{encoder} {(train_counts['encoder'] == e).sum()} ({(train_counts['intra'][train_counts['encoder'] == e]).sum()} I)"
    for e, encoder in enumerate(manifest["encoders"])))

x264 = (val["encoder"] == 0)
for kind, is_intra in (("P", False), ("I", True)):
    gap = (val["mean_qp"] - val["qp"])[x264 & (val["intra"] == is_intra)]
    print(f"x264 {kind} patches: mean QP minus slice QP {gap.mean():+.2f} (|gap| {np.abs(gap).mean():.2f}, 90th % {np.quantile(np.abs(gap), 0.9):.2f})")

## 4. What it says

**34,364 training and 9,628 validation patches**, half from each line (x264 17,271, NVENC 17,093 in training), about
one in seven from an I frame. Every bin of the slice QP has patches from both lines up to 46; NVENC seldom codes above
42 even at 0.25 Mbit/s, so its top bins are thin (64 P patches at 46-52 in validation, none from I frames).

**The same slice QP is less damage from NVENC than from x264.** Bicubic against the target, P frames:

| slice QP | x264 | NVENC |
| --- | --- | --- |
| 0-18 | 0.9569 | 0.9553 |
| 30-34 | 0.8789 | 0.9122 |
| 38-42 | 0.8314 | 0.8641 |
| 42-46 | 0.7994 | 0.8396 |

From QP 30 up NVENC's frames are 0.03-0.04 SSIM-Y closer to the target at the same QP: its loop filter smooths the
block edges x264's `ultrafast` line leaves, and its rate control spends its bits differently. A model told only the
QP sees the two as one; the loop filter flag is what tells them apart.

**On x264's patches the mean QP sits +0.74 above the slice QP on P frames** (90th percentile 2.04), as measured on the
host's lines before notebook 21, and +2.06 on I frames (90th percentile 5.45). Labelled with the slice QP, the model
learns that gap rather than the client correcting for it.